# LeetCode 1922 — Count Good Numbers bằng Python

Link đề: https://leetcode.com/problems/count-good-numbers/ · Độ khó: **Medium** · Chủ đề: Math, Recursion (Fast Exponentiation)

Skeleton LeetCode:

```python
class Solution:
    def countGoodNumbers(self, n: int) -> int:
        ...
```

Lộ trình: **liệt kê mọi chuỗi chữ số** `O(10ⁿ · n)` → **DP / nhân dồn theo từng vị trí** `O(n)` → **công thức `5^⌈n/2⌉ · 4^⌊n/2⌋`** + **lũy thừa nhanh** `O(log n)` (lời giải nộp, tự cài và dùng `pow(a, e, m)`) → **chia để trị đệ quy** trên `n`.

## 1. Mục tiêu học

1. Đếm bằng **quy tắc nhân**: các vị trí độc lập ⇒ số cách = tích số lựa chọn từng vị trí.
2. Đếm đúng số vị trí **chẵn** (`⌈n/2⌉`) và **lẻ** (`⌊n/2⌋`) trong chuỗi đánh số từ 0.
3. Cài **lũy thừa nhanh (binary exponentiation)** lặp và đệ quy, hiểu vì sao chỉ `O(log e)` phép nhân; biết hàm có sẵn `pow(a, e, m)`.
4. Áp dụng **modulo** đúng chỗ (sau mỗi phép nhân) để số không phình to.
5. Kiểm chứng công thức bằng liệt kê với `n` nhỏ và giá trị mẫu của đề (`n = 50`).

## 2. Đề bài & ví dụ

Một chuỗi chữ số là **tốt** nếu các chữ số ở **chỉ số chẵn** (đánh số từ 0) là số **chẵn**, và các chữ số ở **chỉ số lẻ** là số **nguyên tố** (`2, 3, 5, 7`).

Ví dụ `"2582"` tốt (vị trí 0, 2 là `2`, `8` — chẵn; vị trí 1, 3 là `5`, `2` — nguyên tố). `"3245"` không tốt vì `3` ở vị trí chẵn mà không chẵn.

Cho `n`, trả về số chuỗi chữ số tốt độ dài `n`, **modulo `10⁹ + 7`**. Chuỗi chữ số có thể có **số 0 ở đầu**.

**Ví dụ 1:** `n = 1` → `5` (`"0"`, `"2"`, `"4"`, `"6"`, `"8"`).

**Ví dụ 2:** `n = 4` → `400`.

**Ví dụ 3:** `n = 50` → `564908303`.

**Ràng buộc:** `1 ≤ n ≤ 10¹⁵`.

In [1]:
import random
import time
from itertools import product
from typing import List

MOD = 10**9 + 7
EVEN_DIGITS = "02468"
PRIME_DIGITS = "2357"
examples = [(1, 5), (4, 400), (50, 564908303)]

def is_good(s):
    return all((c in EVEN_DIGITS) if i % 2 == 0 else (c in PRIME_DIGITS) for i, c in enumerate(s))

print([is_good(s) for s in ["2582", "3245", "0", "00", "02"]])

[True, False, True, False, True]


## 3. Kiến thức nền

### 3.1 Quy tắc nhân

Nếu vị trí 1 có `a` lựa chọn, vị trí 2 có `b` lựa chọn **không phụ thuộc** vị trí 1, … thì tổng số chuỗi là `a · b · …`.

In [2]:
choices = [len(EVEN_DIGITS) if i % 2 == 0 else len(PRIME_DIGITS) for i in range(4)]
prod = 1
for c in choices:
    prod *= c
print("số lựa chọn từng vị trí:", choices, "-> tích =", prod)

số lựa chọn từng vị trí: [5, 4, 5, 4] -> tích = 400


### 3.2 Số vị trí chẵn / lẻ trong `0..n−1`

Vị trí chẵn: `0, 2, 4, …` — có `⌈n/2⌉ = (n + 1) // 2`. Vị trí lẻ: `⌊n/2⌋ = n // 2`.

In [3]:
for n in range(1, 7):
    ev = sum(1 for i in range(n) if i % 2 == 0)
    print(n, "chẵn:", ev, (n + 1) // 2, "| lẻ:", n - ev, n // 2)

1 chẵn: 1 1 | lẻ: 0 0
2 chẵn: 1 1 | lẻ: 1 1
3 chẵn: 2 2 | lẻ: 1 1
4 chẵn: 2 2 | lẻ: 2 2
5 chẵn: 3 3 | lẻ: 2 2
6 chẵn: 3 3 | lẻ: 3 3


### 3.3 Lũy thừa nhanh (binary exponentiation)

Viết số mũ ở dạng nhị phân: `a^13 = a^8 · a^4 · a^1` (vì `13 = 1101₂`). Bình phương liên tiếp `a, a², a⁴, a⁸, …` và nhân vào kết quả khi bit tương ứng bằng 1 ⇒ `O(log e)` phép nhân.

In [4]:
def fast_pow(a, e, m):
    result, base = 1, a % m
    while e:
        if e & 1:
            result = result * base % m
        base = base * base % m
        e >>= 1
    return result

print(fast_pow(3, 13, 1000), pow(3, 13, 1000), 3 ** 13 % 1000)

323 323 323


### 3.4 `pow(a, e, m)` có sẵn

Hàm dựng sẵn của Python tính `a^e mod m` bằng lũy thừa nhanh (viết bằng C), chạy tức thời kể cả `e = 10¹⁵`.

In [5]:
t = time.perf_counter(); r = pow(5, 10**15, MOD); dt = time.perf_counter() - t
print(r, f"trong {dt*1e6:.0f} µs")

410143883 trong 42 µs


### 3.5 Modulo với phép nhân

`(x · y) mod m = ((x mod m) · (y mod m)) mod m`. Lấy dư **sau mỗi** phép nhân giữ số luôn `< m²` — nhỏ và nhanh.

In [6]:
x, y = 123456789123, 987654321987
print((x * y) % MOD == ((x % MOD) * (y % MOD)) % MOD)

True


### 3.6 Số mũ ở dạng nhị phân, `bit_length`

`e.bit_length()` = số bit của `e` = số vòng lặp của lũy thừa nhanh. Với `e ≤ 10¹⁵ < 2⁵⁰` chỉ cần ≤ 50 vòng — so với `10¹⁵` phép nhân của cách ngây thơ.

In [7]:
for e in [13, 10**6, 10**15]:
    print(f"e = {e}: nhị phân dài {e.bit_length()} bit, số bit 1 = {bin(e).count('1')}")

e = 13: nhị phân dài 4 bit, số bit 1 = 3
e = 1000000: nhị phân dài 20 bit, số bit 1 = 7
e = 1000000000000000: nhị phân dài 50 bit, số bit 1 = 20


## 4. Ý tưởng nền tảng / trực giác

`n = 4`: vị trí `0, 1, 2, 3`.

| vị trí | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| loại | chẵn | lẻ | chẵn | lẻ |
| được dùng | 0,2,4,6,8 | 2,3,5,7 | 0,2,4,6,8 | 2,3,5,7 |
| số lựa chọn | 5 | 4 | 5 | 4 |

Các vị trí độc lập ⇒ `5 · 4 · 5 · 4 = 400`. ✔ Tổng quát: `5^(số vị trí chẵn) · 4^(số vị trí lẻ) = 5^⌈n/2⌉ · 4^⌊n/2⌋`.

Vấn đề duy nhất: `n` tới `10¹⁵` ⇒ không thể nhân `n` lần ⇒ **lũy thừa nhanh**.

In [8]:
for n in [1, 2, 3, 4, 5]:
    print(n, "->", 5 ** ((n + 1) // 2) * 4 ** (n // 2))

1 -> 5
2 -> 20
3 -> 100
4 -> 400
5 -> 2000


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi chuỗi chữ số

In [9]:
def count_brute(n):
    return sum(is_good(s) for s in map("".join, product("0123456789", repeat=n))) % MOD

for n, want in examples[:2]:
    print(n, count_brute(n), "| kỳ vọng", want)

1 5 | kỳ vọng 5
4 400 | kỳ vọng 400


**Độ phức tạp:** `10ⁿ` chuỗi × `O(n)` kiểm ⇒ `O(10ⁿ · n)`. `n = 6` đã một triệu chuỗi; `n = 50` là bất khả.

**Nút thắt:** liệt kê từng chuỗi trong khi các vị trí **độc lập** ⇒ chỉ cần **nhân** số lựa chọn.

In [10]:
t = time.perf_counter(); r = count_brute(6); dt = time.perf_counter() - t
print(f"n=6: {r} trong {dt:.2f}s; mỗi chữ số thêm nhân 10 thời gian")

n=6: 8000 trong 0.29s; mỗi chữ số thêm nhân 10 thời gian


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (quy tắc nhân):** điều kiện của mỗi vị trí chỉ phụ thuộc **chính nó** ⇒ đáp án `5^⌈n/2⌉ · 4^⌊n/2⌋ mod p`.

**Insight 2 (lũy thừa nhanh):** `a^e` với `e ~ 10¹⁵` cần ~50 lần bình phương + ≤ 50 lần nhân. Hoặc gộp: `5^⌈n/2⌉ · 4^⌊n/2⌋ = 20^⌊n/2⌋ · (5 nếu n lẻ)`.

**Quy tắc:** *nếu* bit thấp nhất của `e` là 1 *thì* nhân `base` vào kết quả; luôn bình phương `base` và dịch phải `e`.

**Invariant lũy thừa nhanh:** `result · base^e ≡ a^{e₀} (mod m)` sau mỗi vòng (với `e₀` là số mũ ban đầu).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Nhân dồn theo từng vị trí (DP `O(n)`)

In [11]:
def count_linear(n):
    ans = 1
    for i in range(n):
        ans = ans * (5 if i % 2 == 0 else 4) % MOD
    return ans

for n, want in examples:
    print(n, count_linear(n), "| kỳ vọng", want)

1 5 | kỳ vọng 5
4 400 | kỳ vọng 400
50 564908303 | kỳ vọng 564908303


**Độ phức tạp:** `O(n)` — với `n = 10¹⁵` cần `10¹⁵` vòng (hàng chục năm). Nút thắt: nhân lặp lại cùng một thừa số ⇒ lũy thừa nhanh.

### 7.2 Cách 3 — Công thức + lũy thừa nhanh tự cài (`O(log n)`)

In [12]:
def fast_pow_demo(a, e, m):
    result, base = 1, a % m
    step = 0
    while e:
        bit = e & 1
        if bit:
            result = result * base % m
        print(f"bước {step}: bit={bit}, base={base}, result={result}")
        base = base * base % m
        e >>= 1; step += 1
    return result

print("5^7 mod p:", fast_pow_demo(5, 7, MOD), "| kiểm:", 5 ** 7 % MOD)

bước 0: bit=1, base=5, result=5
bước 1: bit=1, base=25, result=125
bước 2: bit=1, base=625, result=78125
5^7 mod p: 78125 | kiểm: 78125


In [13]:
class SolutionFastPow:
    def countGoodNumbers(self, n: int) -> int:
        return fast_pow(5, (n + 1) // 2, MOD) * fast_pow(4, n // 2, MOD) % MOD

for n, want in examples:
    print(n, SolutionFastPow().countGoodNumbers(n), want)

1 5 5
4 400 400
50 564908303 564908303


**Độ phức tạp:** mỗi `fast_pow` `O(log e)` ≈ 50 vòng ⇒ `O(log n)` thời gian, `O(1)` bộ nhớ.

### 7.3 Cách 4 — `pow` có sẵn và gộp `20^⌊n/2⌋`

In [14]:
class SolutionBuiltin:
    def countGoodNumbers(self, n: int) -> int:
        ans = pow(20, n // 2, MOD)
        if n % 2:
            ans = ans * 5 % MOD
        return ans

for n, want in examples:
    print(n, SolutionBuiltin().countGoodNumbers(n), want)

1 5 5
4 400 400
50 564908303 564908303


### 7.4 Cách 5 — Chia để trị đệ quy trên `n`

Ghép mỗi vị trí chẵn với vị trí lẻ ngay sau nó thành một **cặp** (20 lựa chọn mỗi cặp); có `k = n // 2` cặp và nếu `n` lẻ thì thừa một vị trí chẵn cuối (5 lựa chọn). Đặt `g(k) = 20^k`: chia đôi số cặp, `g(k) = g(k // 2)² · (20 nếu k lẻ)`. (Chia đôi **chuỗi** trực tiếp thì phiền hơn vì nửa sau có thể bắt đầu ở vị trí lẻ.) Độ sâu đệ quy chỉ `log₂ k ≈ 50`.

In [15]:
def pow20(k):
    if k == 0:
        return 1
    half = pow20(k // 2)
    res = half * half % MOD
    return res * 20 % MOD if k % 2 else res

class SolutionDivide:
    def countGoodNumbers(self, n: int) -> int:
        return pow20(n // 2) * (5 if n % 2 else 1) % MOD

for n, want in examples:
    print(n, SolutionDivide().countGoodNumbers(n), want)
print("n = 10^15:", SolutionDivide().countGoodNumbers(10**15))

1 5 5
4 400 400
50 564908303 564908303
n = 10^15: 711414395


### 7.5 Cách 6 — Lũy thừa nhanh "từ trái sang phải"

Biến thể khác của cùng ý tưởng: đọc bit của số mũ **từ cao xuống thấp**; mỗi bước bình phương kết quả, gặp bit 1 thì nhân thêm `a`. (Đây là cách tính tay quen thuộc: `a^13`: `1 → a → a³ → a⁶ → a¹³` theo các bit `1, 1, 0, 1`.)

In [16]:
def fast_pow_ltr(a, e, m, verbose=False):
    result = 1 % m
    for bit in bin(e)[2:]:                    # từ bit cao nhất
        result = result * result % m
        if bit == "1":
            result = result * a % m
        if verbose:
            print(f"bit {bit}: result = {result}")
    return result

print("5^13:", fast_pow_ltr(5, 13, 10**12, verbose=True), "| kiểm:", 5 ** 13)

class SolutionLTR:
    def countGoodNumbers(self, n: int) -> int:
        return fast_pow_ltr(5, (n + 1) // 2, MOD) * fast_pow_ltr(4, n // 2, MOD) % MOD

for n, want in examples:
    print(n, SolutionLTR().countGoodNumbers(n), want)

bit 1: result = 5
bit 1: result = 125
bit 0: result = 15625
bit 1: result = 1220703125
5^13: 1220703125 | kiểm: 1220703125
1 5 5
4 400 400
50 564908303 564908303


Cả hai chiều đều `O(log e)` phép nhân. Bản "trái sang phải" luôn nhân với **cùng** cơ số `a` (có lợi khi `a` nhỏ hoặc phép nhân với `a` rẻ); bản "phải sang trái" (Cách 3) không cần biết trước độ dài số mũ.

## 8. TRACE từng bước

Trace lũy thừa nhanh `5^13 mod p` (`13 = 1101₂`):

| bước | `e` (nhị phân) | bit thấp | `result` | `base` trước khi bình phương |
|---|---|---|---|---|
| 0 | 1101 | 1 | 5 | 5 |
| 1 | 110 | 0 | 5 | 25 |
| 2 | 11 | 1 | 5·625 = 3125 | 625 |
| 3 | 1 | 1 | 3125·390625 = 5¹³ | 390625 |

Kết quả `5¹³ = 1220703125 ≡ 220703118 (mod 10⁹+7)`.

In [17]:
r = fast_pow_demo(5, 13, MOD)
print("kiểm:", r == 5 ** 13 % MOD, 5 ** 13)

bước 0: bit=1, base=5, result=5
bước 1: bit=0, base=25, result=5
bước 2: bit=1, base=625, result=3125
bước 3: bit=1, base=390625, result=220703118
kiểm: True 1220703125


Trace công thức cho từng `n` nhỏ (số vị trí chẵn/lẻ và tích):

In [18]:
def trace_formula(n):
    ev, od = (n + 1) // 2, n // 2
    val = pow(5, ev, MOD) * pow(4, od, MOD) % MOD
    print(f"n={n}: {ev} vị trí chẵn (5 lựa chọn), {od} vị trí lẻ (4 lựa chọn) -> 5^{ev}·4^{od} = {val}")
    return val

for n in [1, 2, 3, 4, 7]:
    trace_formula(n)

n=1: 1 vị trí chẵn (5 lựa chọn), 0 vị trí lẻ (4 lựa chọn) -> 5^1·4^0 = 5
n=2: 1 vị trí chẵn (5 lựa chọn), 1 vị trí lẻ (4 lựa chọn) -> 5^1·4^1 = 20
n=3: 2 vị trí chẵn (5 lựa chọn), 1 vị trí lẻ (4 lựa chọn) -> 5^2·4^1 = 100
n=4: 2 vị trí chẵn (5 lựa chọn), 2 vị trí lẻ (4 lựa chọn) -> 5^2·4^2 = 400
n=7: 4 vị trí chẵn (5 lựa chọn), 3 vị trí lẻ (4 lựa chọn) -> 5^4·4^3 = 40000


## 9. Code nộp LeetCode

In [19]:
class Solution:
    def countGoodNumbers(self, n: int) -> int:
        MOD = 10**9 + 7
        even_pos = (n + 1) // 2          # chỉ số chẵn: 5 lựa chọn (0, 2, 4, 6, 8)
        odd_pos = n // 2                 # chỉ số lẻ: 4 lựa chọn (2, 3, 5, 7)
        return pow(5, even_pos, MOD) * pow(4, odd_pos, MOD) % MOD

for n, want in examples:
    assert Solution().countGoodNumbers(n) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Đếm.** Một chuỗi độ dài `n` là tốt ⇔ với mọi `i`, chữ số `sᵢ ∈ E = {0,2,4,6,8}` nếu `i` chẵn, `sᵢ ∈ Q = {2,3,5,7}` nếu `i` lẻ. Tập các chuỗi tốt là **tích Descartes** `E × Q × E × Q × …` (độ dài `n`) ⇒ kích thước `|E|^{#chẵn} · |Q|^{#lẻ} = 5^⌈n/2⌉ · 4^⌊n/2⌋`.

**Lũy thừa nhanh.** Bất biến `result · base^e ≡ a^{e₀}`: ban đầu `result = 1, base = a, e = e₀`. Nếu `e` lẻ: `result·base · base^{e−1}` giữ nguyên tích; sau đó `base² ` và `e // 2`: `base^{e} = (base²)^{e//2}` khi `e` chẵn. Kết thúc `e = 0` ⇒ `result ≡ a^{e₀}`. Lấy dư sau mỗi phép nhân không đổi giá trị mod `m`.

**Modulo cuối:** tích hai số `< p` rồi `% p` cho đúng `(5^a · 4^b) mod p`.

Kiểm chứng bất biến `result · base^e ≡ a^{e₀}` sau **mỗi** vòng của lũy thừa nhanh, trên nhiều bộ ngẫu nhiên:

In [20]:
rng_i = random.Random(99)
for _ in range(300):
    a, e0, m = rng_i.randint(0, 10**6), rng_i.randint(0, 10**9), rng_i.choice([MOD, 1009, 7])
    target = pow(a, e0, m)
    result, base, e = 1, a % m, e0
    while e:
        assert result * pow(base, e, m) % m == target       # bất biến trước mỗi vòng
        if e & 1: result = result * base % m
        base = base * base % m; e >>= 1
    assert result == target
print("bất biến của lũy thừa nhanh đúng ở mọi vòng")

bất biến của lũy thừa nhanh đúng ở mọi vòng


## 11. Phân tích Time / Space

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê | `O(10ⁿ · n)` | `O(n)` |
| 2. Nhân dồn | `O(n)` | `O(1)` |
| 3. Lũy thừa nhanh tự cài | `O(log n)` | `O(1)` |
| 4. `pow` có sẵn | `O(log n)` | `O(1)` |
| 5. Chia để trị | `O(log n)` | `O(log n)` stack |

In [21]:
for name, f in [("Cách 3", SolutionFastPow().countGoodNumbers), ("Cách 4", SolutionBuiltin().countGoodNumbers),
                ("Cách 5", SolutionDivide().countGoodNumbers), ("Nộp", Solution().countGoodNumbers)]:
    t = time.perf_counter()
    for n in [10**15, 10**15 - 1, 123456789012345, 1]:
        f(n)
    print(f"{name}: 4 lần với n tới 10^15 trong {1e6*(time.perf_counter()-t):.0f} µs")
t = time.perf_counter(); count_linear(10**6); dt = time.perf_counter() - t
print(f"Cách 2 với n=10^6: {dt:.2f}s -> n=10^15 ước tính ≈ {dt * 1e9 / 3600 / 24 / 365:.0f} năm")

Cách 3: 4 lần với n tới 10^15 trong 29 µs
Cách 4: 4 lần với n tới 10^15 trong 8 µs
Cách 5: 4 lần với n tới 10^15 trong 20 µs
Nộp: 4 lần với n tới 10^15 trong 12 µs
Cách 2 với n=10^6: 0.05s -> n=10^15 ước tính ≈ 2 năm


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [22]:
sols = [Solution(), SolutionFastPow(), SolutionBuiltin(), SolutionDivide(), SolutionLTR()]
edge = examples + [(2, 20), (3, 100), (5, 2000), (6, 8000)]
for n, want in edge:
    if n <= 6:
        assert count_brute(n) == want
    assert count_linear(n) == want
    for s in sols:
        assert s.countGoodNumbers(n) == want, (n, s)
    print("OK", n, "->", want)

OK 1 -> 5
OK 4 -> 400
OK 50 -> 564908303
OK 2 -> 20
OK 3 -> 100
OK 5 -> 2000


OK 6 -> 8000


(`n = 2, 3, 5, 6` là giá trị tự dựng — đã đối chiếu bằng liệt kê `count_brute`.)

### 12.2 Đối chiếu với nhân dồn `O(n)` trên nhiều `n` ngẫu nhiên vừa phải, và giữa các cách trên `n` khổng lồ

In [23]:
rng = random.Random(1922)
for _ in range(200):
    n = rng.randint(1, 5000)
    want = count_linear(n)
    for s in sols:
        assert s.countGoodNumbers(n) == want
for _ in range(500):
    n = rng.randint(1, 10**15)
    assert len({s.countGoodNumbers(n) for s in sols}) == 1
print("200 n ≤ 5000 (so nhân dồn) + 500 n tới 10^15 (các cách khớp nhau)")

200 n ≤ 5000 (so nhân dồn) + 500 n tới 10^15 (các cách khớp nhau)


### 12.3 Kiểm `fast_pow` với `pow` trên đủ loại cơ số/số mũ

In [24]:
rng = random.Random(7)
for _ in range(2000):
    a = rng.randint(0, 10**12); e = rng.randint(0, 10**18); m = rng.choice([MOD, 998244353, 97, 2, 1])
    assert fast_pow(a, e, m) == pow(a, e, m)
print("fast_pow khớp pow")

fast_pow khớp pow


### 12.4 Kiểm Ví dụ 3 bằng số nguyên lớn chính xác

`n = 50` nhỏ nên tính được `5²⁵ · 4²⁵ = 20²⁵` **chính xác** bằng số nguyên Python rồi mới lấy dư — một phép kiểm độc lập với lũy thừa nhanh.

In [25]:
exact = 5 ** 25 * 4 ** 25
print(exact, "=", 20 ** 25 == exact, "| mod p =", exact % MOD)
assert exact % MOD == 564908303 == Solution().countGoodNumbers(50)

335544320000000000000000000000000 = True | mod p = 564908303


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đảo vai trò chẵn/lẻ (dùng `4^⌈n/2⌉ · 5^⌊n/2⌋`)

In [26]:
def wrong_swap(n):
    return pow(4, (n + 1) // 2, MOD) * pow(5, n // 2, MOD) % MOD     # SAI

for n, want in examples:
    print(n, "sai:", wrong_swap(n), "| đúng:", want)

1 sai: 4 | đúng: 5
4 sai: 400 | đúng: 400
50 sai: 564908303 | đúng: 564908303


**Vì sao sai:** vị trí **0** là vị trí chẵn ⇒ dùng chữ số chẵn (5 lựa chọn). Với `n` lẻ có **nhiều hơn** một vị trí chẵn so với lẻ. Với `n` chẵn hai cách trùng nhau — nên Ví dụ 2 (`n = 4`) **không** bắt được lỗi, Ví dụ 1 thì bắt được.

### Lỗi 2 — Nhân dồn `n` lần (đúng nhưng TLE)

In [27]:
t = time.perf_counter(); r = count_linear(2 * 10**6); dt = time.perf_counter() - t
print(f"n = 2·10^6: {dt:.2f}s; với n = 10^15 cần ≈ {dt * 5e8 / 3600 / 24 / 365:.0f} năm")
print("lũy thừa nhanh:", Solution().countGoodNumbers(2 * 10**6) == r)

n = 2·10^6: 0.11s; với n = 10^15 cần ≈ 2 năm
lũy thừa nhanh: True


**Vì sao sai:** `n` tới `10¹⁵`. **Sửa:** lũy thừa nhanh `O(log n)`.

### Lỗi 3 — Tính `5**a * 4**b` rồi mới lấy dư

In [28]:
n = 2 * 10**6
t = time.perf_counter(); big = 5 ** ((n + 1) // 2) * 4 ** (n // 2); r1 = big % MOD; t1 = time.perf_counter() - t
t = time.perf_counter(); r2 = Solution().countGoodNumbers(n); t2 = time.perf_counter() - t
print(f"số nguyên khổng lồ có {big.bit_length()} bit, mất {1000*t1:.0f} ms | pow có mod: {1e6*t2:.0f} µs | bằng nhau: {r1 == r2}")

số nguyên khổng lồ có 4321929 bit, mất 86 ms | pow có mod: 27 µs | bằng nhau: True


**Vì sao sai:** kết quả đúng (Python có số nguyên lớn) nhưng `5^(5·10¹⁴)` có cỡ `10¹⁵` bit — không đủ bộ nhớ. Ngay với `n = 2·10⁵` đã chậm hơn hàng nghìn lần. **Sửa:** `pow(a, e, MOD)` — lấy dư trong lúc tính.

### Lỗi 4 — Dùng `math.pow` (số thực)

In [29]:
import math
n = 50
approx = int(math.pow(5, (n + 1) // 2) * math.pow(4, n // 2)) % MOD
print("math.pow:", approx, "| đúng:", Solution().countGoodNumbers(n))

math.pow: 836864072 | đúng: 564908303


**Vì sao sai:** `math.pow` trả về `float` (≈ 15–16 chữ số có nghĩa); `5²⁵ · 4²⁵ ≈ 10³²` đã mất chính xác nên lấy dư cho kết quả vô nghĩa. **Sửa:** số nguyên + `pow(…, …, MOD)`.

### Lỗi 5 — Tập chữ số nguyên tố sai (thêm `1` hoặc `9`)

In [30]:
def wrong_primes(n):
    return pow(5, (n + 1) // 2, MOD) * pow(5, n // 2, MOD) % MOD     # SAI: coi {1,2,3,5,7} là nguyên tố

for n, want in examples:
    print(n, "sai:", wrong_primes(n), "| đúng:", want)

1 sai: 5 | đúng: 5
4 sai: 625 | đúng: 400
50 sai: 876125953 | đúng: 564908303


**Vì sao sai:** `1` **không** phải số nguyên tố; `9 = 3 · 3`. Chữ số nguyên tố chỉ có `2, 3, 5, 7` (4 chữ số).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm số chuỗi thỏa điều kiện từng vị trí độc lập"** ⇒ quy tắc nhân ⇒ thường ra **lũy thừa**.
- `n` cực lớn (`10¹⁵`, `10¹⁸`) ⇒ **lũy thừa nhanh** hoặc **lũy thừa ma trận** (khi các vị trí phụ thuộc nhau theo kiểu chuỗi Markov — Bài tập 5).
- Python: `pow(a, e, m)` là lũy thừa nhanh có sẵn — dùng ngay.
- Đếm vị trí chẵn trong `0..n−1`: `(n + 1) // 2`.
- Kiểm công thức bằng liệt kê với `n ≤ 6`.

## 15. Biến thể của bài

1. **Tập cho phép khác nhau theo vị trí mod `k`** ⇒ tích `Π |S_r|^{số vị trí ≡ r}`.
2. **Thêm ràng buộc giữa hai chữ số kề nhau** (ví dụ không được bằng nhau) ⇒ DP trạng thái "chữ số cuối" + **lũy thừa ma trận** (Bài tập 5).
3. **LeetCode 50 — Pow(x, n):** cài lũy thừa nhanh với số mũ âm.
4. **LeetCode 372 — Super Pow:** số mũ là số rất lớn cho dưới dạng mảng chữ số.
5. **Đếm số nguyên (không có số 0 đầu)** ⇒ vị trí đầu bỏ chữ số `0`: `4 · 5^{⌈n/2⌉−1} · 4^{⌊n/2⌋}`.

Minh họa biến thể 1: tập cho phép **theo vị trí mod `k`** — ví dụ `k = 3` với tập `S₀ = {0..9}`, `S₁ = {1,3,5,7,9}`, `S₂ = {0}` — đáp án `Π_r |S_r|^{c_r}` với `c_r` = số vị trí `i < n` có `i mod k = r`. (Ví dụ tự dựng, đối chiếu bằng liệt kê.)

In [31]:
def count_periodic(n, sets):
    k = len(sets)
    ans = 1
    for r, S in enumerate(sets):
        c = (n - r + k - 1) // k if n > r else 0           # số i in [0, n) với i % k == r
        ans = ans * pow(len(S), c, MOD) % MOD
    return ans

def count_periodic_brute(n, sets):
    k = len(sets)
    return sum(all(s[i] in sets[i % k] for i in range(n)) for s in map("".join, product("0123456789", repeat=n))) % MOD

sets3 = ["0123456789", "13579", "0"]
for n in range(1, 6):
    assert count_periodic(n, sets3) == count_periodic_brute(n, sets3)
assert all(count_periodic(n, [EVEN_DIGITS, PRIME_DIGITS]) == Solution().countGoodNumbers(n) for n in range(1, 200))
print("biến thể tập tuần hoàn: OK", [count_periodic(n, sets3) for n in range(1, 6)])

biến thể tập tuần hoàn: OK [10, 50, 50, 500, 2500]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê | thử mọi chuỗi | `O(10ⁿ n)` | `O(n)` | kiểm chứng `n ≤ 6` |
| 2. Nhân dồn | DP theo vị trí | `O(n)` | `O(1)` | `n ≤ 10⁷` |
| 3. Lũy thừa nhanh tự cài | bình phương liên tiếp | `O(log n)` | `O(1)` | hiểu thuật toán, ngôn ngữ không có `pow` mod |
| 4. `pow` có sẵn | `pow(a, e, m)` | `O(log n)` | `O(1)` | **nộp bài** (Python) |
| 5. Chia để trị | `g(k) = g(k/2)²·…` | `O(log n)` | `O(log n)` | dạng đệ quy |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `countGoodNumbers(2)`, `(3)`, `(5)`?

**Bài 2 (tính tay):** Trace lũy thừa nhanh `4^10 mod 1000`.

**Bài 3 (điền TODO):** Viết `fast_pow_recursive(a, e, m)` bằng đệ quy chia đôi số mũ.

**Bài 4 (tự viết lại):** Viết `count_good_integers(n)` — như bài gốc nhưng chuỗi là **số nguyên** (không có số 0 ở đầu, trừ khi `n = 1` thì `"0"` vẫn tính).

**Bài 5 (mini challenge):** Đếm chuỗi tốt độ dài `n` (mod `p`) với thêm điều kiện **hai chữ số kề nhau phải khác nhau**, `n` tới `10¹⁵` (gợi ý: lũy thừa ma trận).

In [32]:
def fast_pow_recursive(a, e, m):
    # TODO
    pass

def count_good_integers(n):
    # TODO
    pass

def count_good_no_equal_adjacent(n):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `5^⌈n/2⌉·4^⌊n/2⌋` | `n=2`: 5·4 | 20, 100, 2000 |
| 2 | `10 = 1010₂` | Bit 0 = 0, bit 1 = 1 (nhân 16)… | `4^10 = 1048576` ⇒ 576 |
| 3 | `a^e = (a^{e/2})²` (· `a` nếu `e` lẻ) | Trường hợp gốc `e = 0` | Độ sâu `log e` |
| 4 | `n = 1`: 5 (gồm `"0"`) | `n ≥ 2`: vị trí 0 chỉ còn `2,4,6,8` | `4 · 5^{⌈n/2⌉−1} · 4^{⌊n/2⌋}` |
| 5 | Trạng thái = chữ số cuối | Hai ma trận chuyển `A` (vị trí lẻ → chẵn), `B` (chẵn → lẻ) | Lũy thừa `(A·B)^{k}` rồi xử lý phần lẻ |

## 19. Đáp án tham khảo

In [33]:
r1 = [Solution().countGoodNumbers(n) for n in (2, 3, 5)]
print("Bài 1:", r1); assert r1 == [20, 100, 2000]
print("Bài 2:"); r2 = fast_pow_demo(4, 10, 1000); assert r2 == 576

Bài 1: [20, 100, 2000]
Bài 2:
bước 0: bit=0, base=4, result=1
bước 1: bit=1, base=16, result=16
bước 2: bit=0, base=256, result=16
bước 3: bit=1, base=536, result=576


In [34]:
def fast_pow_recursive(a, e, m):
    if e == 0:
        return 1 % m
    half = fast_pow_recursive(a, e // 2, m)
    res = half * half % m
    return res * a % m if e % 2 else res

def count_good_integers(n):
    if n == 1:
        return 5
    return 4 * pow(5, (n + 1) // 2 - 1, MOD) * pow(4, n // 2, MOD) % MOD

def count_good_integers_brute(n):
    lo = 0 if n == 1 else 10 ** (n - 1)
    return sum(is_good(str(x)) for x in range(lo, 10 ** n))

rng = random.Random(3)
for _ in range(500):
    a, e, m = rng.randint(0, 10**9), rng.randint(0, 10**15), rng.choice([MOD, 13, 1000])
    assert fast_pow_recursive(a, e, m) == pow(a, e, m)
for n in range(1, 6):
    assert count_good_integers(n) == count_good_integers_brute(n)
print("Bài 3, 4: OK", [count_good_integers(n) for n in range(1, 6)])

Bài 3, 4: OK [5, 16, 80, 320, 1600]


In [35]:
def mat_mul(A, B):
    n, k, m = len(A), len(B), len(B[0])
    C = [[0] * m for _ in range(n)]
    for i in range(n):
        Ai = A[i]
        for t in range(k):
            if Ai[t]:
                a = Ai[t]; Bt = B[t]; Ci = C[i]
                for j in range(m):
                    Ci[j] = (Ci[j] + a * Bt[j]) % MOD
    return C

def mat_pow(M, e):
    size = len(M)
    R = [[int(i == j) for j in range(size)] for i in range(size)]
    while e:
        if e & 1: R = mat_mul(R, M)
        M = mat_mul(M, M); e >>= 1
    return R

def count_good_no_equal_adjacent(n):
    digits = list(range(10))
    E = {0, 2, 4, 6, 8}; Q = {2, 3, 5, 7}
    # T_to_odd[x][y] = 1 nếu chữ số x (ở vị trí chẵn) có thể theo sau bởi y (vị trí lẻ)
    T_to_odd = [[int(x in E and y in Q and x != y) for y in digits] for x in digits]
    T_to_even = [[int(x in Q and y in E and x != y) for y in digits] for x in digits]
    start = [[int(d in E) for d in digits]]                     # vị trí 0
    pairs = (n - 1) // 2                                         # số lần (lẻ, chẵn) đầy đủ sau vị trí 0
    P = mat_pow(mat_mul(T_to_odd, T_to_even), pairs)
    v = mat_mul(start, P)
    if (n - 1) % 2:                                              # còn một vị trí lẻ cuối
        v = mat_mul(v, T_to_odd)
    return sum(v[0]) % MOD

def no_equal_brute(n):
    return sum(is_good(s) and all(s[i] != s[i + 1] for i in range(n - 1))
               for s in map("".join, product("0123456789", repeat=n)))

for n in range(1, 6):
    assert count_good_no_equal_adjacent(n) == no_equal_brute(n), n
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", [count_good_no_equal_adjacent(n) for n in range(1, 6)],
      "| n = 10^15:", count_good_no_equal_adjacent(10**15))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK [5, 19, 91, 349, 1669] | n = 10^15: 467183158


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
MOD = 10**9 + 7
return pow(5, (n + 1) // 2, MOD) * pow(4, n // 2, MOD) % MOD

def fast_pow(a, e, m):            # khi cần tự cài
    r, b = 1, a % m
    while e:
        if e & 1: r = r * b % m
        b = b * b % m; e >>= 1
    return r
```

**Checklist:**
- [ ] Vị trí chẵn (gồm vị trí 0) có **5** lựa chọn, vị trí lẻ có **4** (`2,3,5,7`)?
- [ ] Số vị trí chẵn `(n + 1) // 2`, lẻ `n // 2`?
- [ ] Lũy thừa **nhanh** có modulo (`pow(a, e, MOD)`), không nhân dồn `n` lần?
- [ ] Không dùng `math.pow` / số thực?
- [ ] Kiểm `n = 1 → 5`, `n = 4 → 400`, `n = 50 → 564908303`.